# Chunking Strategies Demo

Compare different chunking approaches on real documents.

**Strategies covered:**
1. Fixed-size chunking
2. Recursive character text splitting  
3. Semantic chunking (embedding-based boundaries)

**Prerequisites:**
- `pip install langchain-text-splitters sentence-transformers`
- Ollama running with `nomic-embed-text` model

```bash
ollama pull nomic-embed-text
```

In [1]:
# Setup and sample document
import subprocess
import sys

def check_ollama():
    """Verify Ollama is running."""
    try:
        result = subprocess.run(["ollama", "list"], capture_output=True, text=True, timeout=5)
        if result.returncode == 0:
            print("✓ Ollama is running")
            if "nomic-embed-text" in result.stdout:
                print("✓ nomic-embed-text model available")
            else:
                print("⚠ Run: ollama pull nomic-embed-text")
            return True
    except Exception as e:
        print(f"✗ Ollama not available: {e}")
        return False

check_ollama()

# Sample document for chunking experiments
SAMPLE_DOCUMENT = """
Remote Work Policy

1. Overview
The company's remote work policy enables employees to work from locations other than the primary office. This policy applies to all full-time employees who have completed their probation period of 90 days.

2. Eligibility
Employees in the following categories are eligible for remote work:
- Software engineers and developers
- Product managers and designers
- Marketing and sales professionals
- Customer support specialists (with manager approval)

Employees in facilities, security, and front-desk roles are not eligible for remote work due to the nature of their responsibilities.

3. Work Schedule
Remote employees must maintain core hours from 10:00 AM to 3:00 PM in their local timezone. Outside of core hours, employees have flexibility in scheduling their remaining work hours. All employees must be available for team meetings and respond to urgent communications within 30 minutes during business hours.

4. Equipment and Expenses
The company provides a laptop and necessary software licenses. Employees receive a monthly stipend of $100 for internet and home office expenses. Additional equipment purchases require manager approval.

5. Performance Expectations
Remote employees are held to the same performance standards as office-based employees. Managers will conduct monthly check-ins to discuss progress and address any concerns. Performance reviews occur quarterly and follow the standard company process.

6. Security Requirements
All remote employees must:
- Use company-provided VPN for accessing internal systems
- Enable two-factor authentication on all accounts
- Not use public WiFi for sensitive work without VPN
- Report any security incidents immediately

7. Policy Violations
Violations of this policy may result in revocation of remote work privileges. Repeated violations may lead to disciplinary action up to and including termination.
"""

print(f"Sample document: {len(SAMPLE_DOCUMENT)} characters")
print(f"Approximate tokens: ~{len(SAMPLE_DOCUMENT)//4}")

✓ Ollama is running
✓ nomic-embed-text model available
Sample document: 1901 characters
Approximate tokens: ~475


---

## 1. Fixed-Size Chunking

The simplest approach: split every N characters with optional overlap.

In [2]:
def fixed_size_chunk(text: str, chunk_size: int = 300, overlap: int = 50) -> list[str]:
    """
    Split text into fixed-size chunks with overlap.
    
    Problems:
    - Breaks mid-sentence, mid-word
    - No awareness of document structure
    - Fragments lose context
    """
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        start = end - overlap
    return chunks

# Apply fixed-size chunking
fixed_chunks = fixed_size_chunk(SAMPLE_DOCUMENT, chunk_size=300, overlap=50)

print(f"Fixed-Size Chunking (300 chars, 50 overlap)")
print("=" * 60)
print(f"Total chunks: {len(fixed_chunks)}\n")

for i, chunk in enumerate(fixed_chunks[:4]):
    print(f"Chunk {i+1} ({len(chunk)} chars):")
    print(f"  \"{chunk[:80]}...\"")
    # Check if chunk breaks mid-sentence
    if not chunk.rstrip().endswith(('.', '!', '?', ':')):
        print("  ⚠ BREAKS MID-SENTENCE")
    print()

Fixed-Size Chunking (300 chars, 50 overlap)
Total chunks: 8

Chunk 1 (299 chars):
  "Remote Work Policy

1. Overview
The company's remote work policy enables employe..."
  ⚠ BREAKS MID-SENTENCE

Chunk 2 (300 chars):
  "lity
Employees in the following categories are eligible for remote work:
- Softw..."
  ⚠ BREAKS MID-SENTENCE

Chunk 3 (300 chars):
  "ilities, security, and front-desk roles are not eligible for remote work due to ..."
  ⚠ BREAKS MID-SENTENCE

Chunk 4 (299 chars):
  "employees have flexibility in scheduling their remaining work hours. All employe..."
  ⚠ BREAKS MID-SENTENCE



---

## 2. Recursive Character Text Splitting

Tries paragraph breaks first, then sentences, then words, then characters.

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Configure recursive splitter
recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=0,  # Chroma research: 0 overlap often best
    separators=["\n\n", "\n", ". ", ", ", " ", ""],
    length_function=len,
)

recursive_chunks = recursive_splitter.split_text(SAMPLE_DOCUMENT)

print(f"Recursive Character Splitting (400 chars, 0 overlap)")
print("=" * 60)
print(f"Total chunks: {len(recursive_chunks)}\n")

for i, chunk in enumerate(recursive_chunks[:4]):
    print(f"Chunk {i+1} ({len(chunk)} chars):")
    print(f"  \"{chunk[:100]}...\"")
    # Check completeness
    if chunk.rstrip().endswith(('.', '!', '?', ':')):
        print("  ✓ Complete sentence/section")
    print()

Recursive Character Splitting (400 chars, 0 overlap)
Total chunks: 7

Chunk 1 (237 chars):
  "Remote Work Policy

1. Overview
The company's remote work policy enables employees to work from loca..."
  ✓ Complete sentence/section

Chunk 2 (377 chars):
  "2. Eligibility
Employees in the following categories are eligible for remote work:
- Software engine..."
  ✓ Complete sentence/section

Chunk 3 (328 chars):
  "3. Work Schedule
Remote employees must maintain core hours from 10:00 AM to 3:00 PM in their local t..."
  ✓ Complete sentence/section

Chunk 4 (228 chars):
  "4. Equipment and Expenses
The company provides a laptop and necessary software licenses. Employees r..."
  ✓ Complete sentence/section



---

## 3. Semantic Chunking

Uses embeddings to detect topic boundaries. Splits where semantic similarity drops.

In [4]:
import requests
import numpy as np

def get_ollama_embedding(text: str, model: str = "nomic-embed-text") -> list[float]:
    """Get embedding from Ollama."""
    response = requests.post(
        "http://localhost:11434/api/embeddings",
        json={"model": model, "prompt": text}
    )
    return response.json()["embedding"]

def cosine_similarity(a: list[float], b: list[float]) -> float:
    """Compute cosine similarity between two vectors."""
    a, b = np.array(a), np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def semantic_chunk(
    text: str, 
    threshold: float = 0.75,
    min_chunk_size: int = 100
) -> list[str]:
    """
    Split text based on semantic similarity between adjacent sentences.
    
    When similarity drops below threshold, start a new chunk.
    """
    # Split into sentences first
    import re
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    sentences = [s for s in sentences if len(s) > 10]
    
    if len(sentences) < 2:
        return [text]
    
    # Get embeddings for each sentence
    print("Getting embeddings for sentences...")
    embeddings = [get_ollama_embedding(s) for s in sentences]
    
    # Find breakpoints where similarity drops
    chunks = []
    current_chunk = [sentences[0]]
    
    for i in range(1, len(sentences)):
        similarity = cosine_similarity(embeddings[i-1], embeddings[i])
        
        if similarity < threshold and len(' '.join(current_chunk)) >= min_chunk_size:
            # Low similarity = topic change = new chunk
            chunks.append(' '.join(current_chunk))
            current_chunk = [sentences[i]]
        else:
            current_chunk.append(sentences[i])
    
    # Don't forget the last chunk
    if current_chunk:
        chunks.append(' '.join(current_chunk))
    
    return chunks

# Apply semantic chunking
try:
    semantic_chunks = semantic_chunk(SAMPLE_DOCUMENT, threshold=0.7)
    
    print(f"\nSemantic Chunking (similarity threshold: 0.7)")
    print("=" * 60)
    print(f"Total chunks: {len(semantic_chunks)}\n")
    
    for i, chunk in enumerate(semantic_chunks[:4]):
        print(f"Chunk {i+1} ({len(chunk)} chars):")
        print(f"  \"{chunk[:100]}...\"")
        print()
except Exception as e:
    print(f"Semantic chunking failed: {e}")
    print("Make sure Ollama is running with nomic-embed-text model")

Getting embeddings for sentences...

Semantic Chunking (similarity threshold: 0.7)
Total chunks: 10

Chunk 1 (136 chars):
  "Remote Work Policy

1. Overview
The company's remote work policy enables employees to work from loca..."

Chunk 2 (100 chars):
  "This policy applies to all full-time employees who have completed their probation period of 90 days...."

Chunk 3 (374 chars):
  "Eligibility
Employees in the following categories are eligible for remote work:
- Software engineers..."

Chunk 4 (197 chars):
  "Work Schedule
Remote employees must maintain core hours from 10:00 AM to 3:00 PM in their local time..."



---

## 4. Comparison: Chunk Quality Metrics

In [5]:
def analyze_chunk_quality(chunks: list[str], name: str):
    """Analyze chunk quality metrics."""
    
    # Basic stats
    lengths = [len(c) for c in chunks]
    
    # Count sentence breaks (rough measure of completeness)
    complete_sentences = sum(
        1 for c in chunks 
        if c.rstrip().endswith(('.', '!', '?', ':'))
    )
    
    # Check for mid-word breaks
    mid_word_breaks = sum(
        1 for c in chunks 
        if c and not c[0].isupper() and not c[0].isdigit() and c[0] != '-'
    )
    
    print(f"\n{name}")
    print("-" * 40)
    print(f"  Chunks: {len(chunks)}")
    print(f"  Avg length: {np.mean(lengths):.0f} chars")
    print(f"  Min/Max: {min(lengths)}/{max(lengths)} chars")
    print(f"  Complete sentences: {complete_sentences}/{len(chunks)} ({100*complete_sentences/len(chunks):.0f}%)")
    print(f"  Mid-word breaks: {mid_word_breaks}")

print("Chunk Quality Analysis")
print("=" * 60)

analyze_chunk_quality(fixed_chunks, "Fixed-Size (300, 50 overlap)")
analyze_chunk_quality(recursive_chunks, "Recursive (400, 0 overlap)")

try:
    analyze_chunk_quality(semantic_chunks, "Semantic (threshold 0.7)")
except:
    print("\n  Semantic chunks not available")

Chunk Quality Analysis

Fixed-Size (300, 50 overlap)
----------------------------------------
  Chunks: 8
  Avg length: 281 chars
  Min/Max: 150/300 chars
  Complete sentences: 1/8 (12%)
  Mid-word breaks: 7

Recursive (400, 0 overlap)
----------------------------------------
  Chunks: 7
  Avg length: 270 chars
  Min/Max: 183/377 chars
  Complete sentences: 6/7 (86%)
  Mid-word breaks: 0

Semantic (threshold 0.7)
----------------------------------------
  Chunks: 10
  Avg length: 187 chars
  Min/Max: 84/374 chars
  Complete sentences: 10/10 (100%)
  Mid-word breaks: 0


---

## 5. Embedding Coherence Test

A chunk with high internal coherence should have similar embeddings for its parts.

In [6]:
def measure_coherence(chunks: list[str], name: str, sample_size: int = 3):
    """
    Measure embedding coherence within chunks.
    
    Split each chunk in half, embed both halves, measure similarity.
    High similarity = coherent chunk (both halves about same topic).
    """
    coherence_scores = []
    
    for chunk in chunks[:sample_size]:
        if len(chunk) < 100:
            continue
        
        # Split chunk in half
        mid = len(chunk) // 2
        first_half = chunk[:mid]
        second_half = chunk[mid:]
        
        # Get embeddings
        emb1 = get_ollama_embedding(first_half)
        emb2 = get_ollama_embedding(second_half)
        
        # Measure similarity
        similarity = cosine_similarity(emb1, emb2)
        coherence_scores.append(similarity)
    
    if coherence_scores:
        avg_coherence = np.mean(coherence_scores)
        print(f"{name}: avg coherence = {avg_coherence:.3f}")
        return avg_coherence
    return 0.0

try:
    print("Embedding Coherence Test")
    print("=" * 60)
    print("(Higher = more coherent chunks)\n")
    
    measure_coherence(fixed_chunks, "Fixed-Size")
    measure_coherence(recursive_chunks, "Recursive")
    
    try:
        measure_coherence(semantic_chunks, "Semantic")
    except:
        pass
        
except Exception as e:
    print(f"Coherence test failed: {e}")

Embedding Coherence Test
(Higher = more coherent chunks)

Fixed-Size: avg coherence = 0.581
Recursive: avg coherence = 0.701
Semantic: avg coherence = 0.609


---

## Summary

| Strategy | Pros | Cons | Best For |
|----------|------|------|----------|
| Fixed-Size | Simple, fast | Breaks mid-sentence | Logs, uniform data |
| Recursive | Better boundaries | Still arbitrary | General documents |
| Semantic | Topic-aware | Slow, requires embeddings | High-value docs |

**Recommendation:** Start with Recursive (400 tokens, 0 overlap).
Move to Semantic only if retrieval quality issues and indexing cost acceptable.